# 08 · Tópicos v2, agenda y puntos de cambio (S1)
**Proyecto PLN · Entrega 3** — *¿Qué tan estable es la agenda temática de la prensa colombiana entre 1980 y 2011 y cuándo cambia?* Agenda = prevalencia de tópicos por año.

Resuelve las observaciones de la entrega 2:
- **(a)** tópicos incoherentes o que no son temas → se marcan como residuales con su motivo;
- **(b)** K = 40 era excesivo → rejilla 8–20 con una regla declarada antes de ver los resultados;
- **(c)** métricas de desempeño con interpretación;
- **NPMI mal usada** → coherencia C_NPMI con ventana deslizante sobre un **held-out**.

Además, se demuestra si los cambios de la entrega 2 (1995, 1999/2000, 2006) sobreviven al control de formato de El Tiempo.

Código: `agenda.py`. Entrada: `utils.load_corpus_procesado` (07) y `data/lista_blanca_con_propn.txt`. No se vuelve a correr spaCy.

**Celdas costosas (💰)** en local (8 núcleos): comparación de bigramas (~5 min), rejilla de K (~5–10 min), proyección de θ y variantes de robustez (~10 min), nulos y bootstrap (~5 min).

In [ ]:
import json, time, warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.exceptions import ConvergenceWarning

import agenda as ag
import utils

warnings.filterwarnings("ignore", category=ConvergenceWarning)
pd.set_option("display.width", 170)
pd.set_option("display.max_colwidth", 120)

FAST = False
cfg = ag.Config08.rapida() if FAST else ag.Config08()
OUT = utils.results_dir() / ("08_fast" if FAST else "08")
OUT.mkdir(parents=True, exist_ok=True)
TIEMPOS = {}


def fig(nombre):
    plt.tight_layout()
    plt.savefig(OUT / nombre, dpi=120)
    plt.show()


print(cfg)

## 0. Reglas declaradas antes de ver resultados
Estas reglas quedan escritas (y en `cfg`) **antes** de cualquier ajuste. La revisión manual solo puede (i) marcar o desmarcar residuales en el CSV y (ii) mover K a un valor **adyacente**, con el motivo escrito en la celda `K_MANUAL`.

1. **Bigramas.** Se comparan con K = `k_ref` = 14 (centro de la rejilla) y las semillas 0–2. Gana la estrategia con mayor NPMI media held-out. Si otras quedan a ≤ 1 EE de la mejor, se desempata por menor % de residuales y luego por mayor estabilidad. Si la elegida no supera a **A** (sin bigramas) por más de 1 EE, se va sin bigramas.
2. **K.** Es el **K más chico** cuya NPMI media held-out queda a ≤ 1 EE (entre semillas) del mejor K y cuya estabilidad por coseno entre semillas es ≥ 0,80. Si ningún K cumple, se toma el más estable. El modelo final es la **semilla medoide** (la más parecida a las otras dos).
3. **Residual** (marca automática, editable). Un tópico se marca si cumple al menos una condición:
   - C_NPMI < 0,02;
   - más de 3 de sus top-10 son genéricos (df > 15 % o presentes en el top-10 de ≥ 30 % de los tópicos);
   - algún top-10 es día, mes, nombre de pila, metatexto o scraping;
   - estabilidad del tópico < 0,5;
   - formato: prevalencia breve/no breve > 3 (o < 1/3), o |ρ de Spearman con la longitud| > 0,3.

   Los residuales se excluyen de la agenda (θ se renormaliza) y se reportan.
4. **Cambios.** Un cambio es **robusto** si cumple tres condiciones (con ±1 año):
   - aparece en la serie **sin breves**, en la serie **con igual peso por mes** y en la serie con ambos controles a la vez;
   - aparece en ≥ 50 % de las variantes votantes: las otras 2 semillas, K−2, K+2, con/sin bigramas y con/sin lista blanca;
   - además, siempre se informa si coincide con un cambio en las **series de formato**.

   Un cambio que coincide con formato y no sobrevive a los controles es **artefacto probable**. El **consenso** solo incluye los robustos. Semana y Dinero son contraste y no votan.
5. **Tamaño del efecto antes que p-valores.** Con N grande todo resulta significativo. Se reporta el exceso de JS sobre el ruido y su fracción de la distancia entre fuentes; los p-valores con BH van como anexo.

## 1. Diagnóstico de cobertura y formato

In [ ]:
COLS = [
    "id",
    "fuente",
    "fecha",
    "año",
    "mes",
    "titulo",
    "es_breve",
    "es_carta",
    "n_palabras_cuerpo",
    "titulo_capitalizado",
    "lead_duplicado",
    "lemas_contenido",
    "pos_contenido",
    "cortes_contenido",
    "lemas_nom_adj",
]
t0 = time.time()
dfp = utils.load_corpus_procesado(fast=FAST, columns=COLS)
dfp = dfp.dropna(subset=["año"])
dfp = dfp[dfp["año"].between(cfg.año_min, cfg.año_max)].reset_index(drop=True)
dfp["año"] = dfp["año"].astype(int)
dfp["mes"] = dfp["mes"].astype(int)
print(f"{len(dfp):,} documentos | carga {time.time() - t0:.0f} s")
import prep

meta_pq = prep.leer_meta(
    utils.data_dir() / (utils.CORPUS_PROCESADO_FAST if FAST else utils.CORPUS_PROCESADO)
)
print(
    "Parquet: pipeline",
    meta_pq.get("pipeline_version"),
    "| hash nlp",
    meta_pq.get("hash_nlp"),
    "| prep.py:",
    prep.PIPELINE_VERSION,
)
if meta_pq and meta_pq.get("pipeline_version") != prep.PIPELINE_VERSION:
    raise RuntimeError(
        "el parquet no corresponde a la versión actual de prep.py: regenéralo con el 07"
    )

cob = ag.tabla_cobertura(dfp)
fmt = ag.series_formato(dfp)
for f in [ag.ET, ag.SEMANA, ag.DINERO]:
    piv = cob[cob.fuente == f].pivot_table(
        index="año", columns="mes", values="docs", fill_value=0
    )
    print(f"\n=== {ag.CORTO[f]}: documentos por año × mes ===\n{piv.to_string()}")
cob.to_csv(OUT / "cobertura_fuente_año_mes.csv", index=False)

In [ ]:
for col, nombre in [
    ("pct_breve", "% es_breve"),
    ("pct_carta", "% es_carta"),
    ("mediana_palabras", "mediana de palabras del cuerpo"),
    ("mediana_palabras_no_breve", "mediana de palabras (sin breves)"),
    ("pct_titulo_cap", "% titulares en Mayúscula Inicial"),
    ("pct_lead_dup", "% lead duplicado"),
]:
    print(f"\n{nombre} por año y fuente:")
    print(
        fmt.pivot(index="año", columns="fuente", values=col)
        .rename(columns=ag.CORTO)
        .round(1)
        .to_string()
    )
fmt.round(2).to_csv(OUT / "formato_fuente_año.csv", index=False)

**Meses comparables.** Un mes es comparable si tiene al menos `max(10, 0,3 × mediana mensual de su fuente)` documentos. El umbral es relativo porque Semana es semanal y El Tiempo diario: un umbral fijo descartaría casi todo Semana o aceptaría meses casi vacíos de El Tiempo. Un año entra a la serie con **igual peso por mes** si tiene ≥ 6 meses comparables. Su prevalencia es la media de las medias mensuales, así que un año con huecos no queda sesgado hacia los meses con más cobertura.

In [ ]:
print("Docs por mes (meses con ≥1 doc):")
print(
    cob[cob.docs > 0]
    .groupby("fuente")["docs"]
    .describe()
    .rename(index=ag.CORTO)
    .round(0)
    .to_string()
)
UMB = ag.umbrales_mes(cob, cfg)
mv = ag.meses_validos(cob, UMB)
print("\nUmbral de documentos por mes:", UMB.rename(index=ag.CORTO).to_dict())
VALIDO = {
    f: set(zip(s["año"], s["mes"])) for f, s in mv[mv["valido"]].groupby("fuente")
}
nval = mv.groupby(["fuente", "año"])["valido"].sum().unstack(0).rename(columns=ag.CORTO)
print("\nMeses comparables por año:")
print(nval.to_string())
inv = mv[(mv.fuente == ag.ET) & (mv.año >= cfg.principal_desde) & ~mv.valido]
print("\nEl Tiempo: meses NO comparables:")
print(inv[["año", "mes", "docs", "umbral"]].to_string(index=False))
mv.to_csv(OUT / "meses_validos.csv", index=False)

## 2. Corpus de modelado
- **Ajuste:** solo documentos **no breves y no cartas**, de las tres fuentes, con un tope de 1.200 por año (como el balanceo del 06). Así la estructura de tópicos no la dictan las notas de 60 palabras de El Tiempo 1990–2005 ni el volumen de El Tiempo 2000–2011. El IDF también se ajusta aquí.
- **Held-out:** 300 documentos por año de la **misma población**, apartados antes de muestrear el ajuste (como máximo 20 % del año). No entran ni al vectorizador ni al NMF y solo sirven para medir coherencia.
- **Proyección:** θ se calcula para los 91.585 documentos. Las breves y las cartas no se borran: se analizan como **estrato**, con el nulo estratificado y la serie sin breves.

In [ ]:
idx_fit, idx_ho = ag.particion_ajuste_heldout(dfp, cfg)
assert not set(idx_fit) & set(idx_ho)
años, meses = dfp["año"].values, dfp["mes"].values
fuentes, breve = dfp["fuente"].values, dfp["es_breve"].values
tab = (
    pd.DataFrame(
        {
            "ajuste": pd.Series(años[idx_fit]).value_counts(),
            "heldout": pd.Series(años[idx_ho]).value_counts(),
            "total": pd.Series(años).value_counts(),
        }
    )
    .fillna(0)
    .astype(int)
    .sort_index()
)
print(f"Ajuste: {len(idx_fit):,} | held-out: {len(idx_ho):,}")
print(tab.T.to_string())
print(
    "\nFuente en ajuste:",
    pd.Series(fuentes[idx_fit]).map(ag.CORTO).value_counts().to_dict(),
)

In [ ]:
# Verificación: reconstruir desde lemas_contenido + pos + cortes reproduce lemas_nom_adj
rs = np.random.default_rng(0).choice(len(dfp), min(3000, len(dfp)), replace=False)
iguales = [
    ag.reconstruir_tokens(
        dfp.lemas_contenido[i], dfp.pos_contenido[i], dfp.cortes_contenido[i]
    )
    == list(dfp.lemas_nom_adj[i])
    for i in rs
]
assert all(iguales), f"{len(iguales) - sum(iguales)} documentos no coinciden"
print(
    "OK: la reconstrucción sin bigramas reproduce lemas_nom_adj en",
    len(rs),
    "documentos",
)

## 3. Representación
### 3.1 Estados Unidos y plurales sin lematizar
- **Estados Unidos (S0 v3.0.2).** Desde la 3.0.2 (y en la 3.0.3), `lemas_contenido` trae `estados`/P + `unidos`/P, y en la tabla NPMI aparece el bigrama `estados_unidos` (con_propn, n = 17.950). PROPN no entra en `lemas_nom_adj`, así que el país solo existiría en las estrategias con lista blanca, y la comparación de bigramas mezclaría "con o sin lista blanca" con "con o sin Estados Unidos". Para evitarlo, `unidos`/P se renombra siempre a `estados_unidos`. Cuando la lista blanca está activa, el bigrama produce **el mismo token**, así que el país entra igual en todas las estrategias. El mapa de plurales no toca ese token.
- **Plurales.** Se mapea plural → singular **solo si el singular existe** como N/A con df ≥ 5 **y** df(singular) ≥ 0,1·df(plural). La segunda condición evita llevar una palabra frecuente a un lema raro mal lematizado: sin ella, *crisis* (df 4.861) iba a "crisi" (df 5) y *a mediados de* iba a "mediado". Se excluyen las terminaciones invariables (-is, -us, vocal tildada + s: crisis, énfasis, tenis, status, estrés). Si hay varios candidatos (`-s`, `-es`, tilde restituida o quitada, `-ces` → `-z`), gana el de mayor df. La tabla completa queda en `mapa_plurales.csv`; para excluir un par, agrégalo a `EXCLUIR_PLURAL`. Los bigramas se unen **antes** del mapa, con las claves del CSV del 07, y después el mapa se aplica también a sus componentes (`derechos_humanos` → `derecho_humano`).

In [ ]:
EXCLUIR_PLURAL = (
    set(cfg.no_pluralizar) | set()
)  # agrega aquí pares dudosos tras revisar la tabla
dfv = ag.df_lemas(dfp["lemas_nom_adj"])
mp = ag.mapa_plurales(
    dfv, cfg.min_df_plural, excluir=EXCLUIR_PLURAL, ratio_min=cfg.ratio_plural_min
)
MAPA = dict(zip(mp.plural, mp.singular))
mp.to_csv(OUT / "mapa_plurales.csv", index=False)
print(f"{len(mp):,} plurales mapeados (de {len(dfv):,} lemas N/A)")
print(mp.head(40).to_string(index=False))
print("\nMuestra aleatoria para auditar:")
print(mp.sample(min(25, len(mp)), random_state=1).to_string(index=False))

### 3.2 `min_df` y `max_df` a partir de la distribución de frecuencias documentales

In [ ]:
KW = dict(renombres=cfg.renombres_propn, mapa=MAPA)
tok_fitA = ag.tokens_corpus(dfp, idx_fit, **KW)
por_min, por_max, top_df = ag.tabla_df(tok_fitA)
print(f"Documentos de ajuste: {len(tok_fitA):,}\n")
print(por_min.round(2).to_string(index=False))
print()
print(por_max.round(2).to_string(index=False))
print("\nTérminos con mayor df:")
print(top_df.round(3).head(30).to_string())
del tok_fitA

**Lectura y decisión.**
- **`min_df = 10`:** quita la larga cola de términos que aparecen en menos de 10 documentos de ajuste. Son la mayoría del vocabulario, pero cubren una fracción mínima de las ocurrencias. Además, con menos de 10 documentos un término no puede definir un tema ni estimarse con estabilidad.
- **`max_df = 0,30`:** quita los términos presentes en casi un tercio de los documentos (*país, gobierno, colombia…*), que no discriminan temas y eran los que formaban tópicos genéricos en el 06.

Si la tabla de arriba muestra un salto distinto, se cambia aquí:

In [ ]:
# cfg.min_df, cfg.max_df = 10, 0.30   # ← ajustar solo si la tabla lo justifica
print("min_df =", cfg.min_df, "| max_df =", cfg.max_df)

### 3.3 Selección de bigramas por comparación
Primero se recalcula la tabla NPMI **completa** (el CSV del 07 solo trae NPMI ≥ 0,35, así que D y E no saldrían de él). Es el mismo cálculo del 07 (`prep.colocaciones_npmi`, n ≥ 20, df ≥ 10) y no corre spaCy.

| Estrategia | Bigramas |
|---|---|
| A | ninguno |
| B | nom_adj con NPMI ≥ 0,35 |
| C | B con n ≥ 50 |
| D | los N más frecuentes con NPMI ≥ 0,20 |
| E | los N más frecuentes (sin NPMI), como control |

N = |C| en D y E, para que su diferencia aísle el efecto del NPMI. Cada estrategia se prueba con y sin la **lista blanca** (`data/lista_blanca_con_propn.txt`). La lista son sobre todo instituciones `con_propn`, más tres `nom_adj` temáticos agregados a mano: `acuerdo_paz` (NPMI 0,32: el NPMI lo castiga porque *acuerdo* y *paz* son muy frecuentes), `medio_ambiente` y `medio_comunicación`. Los dos últimos evitan que `medio` suelto (rescatado en la 3.0.3) forme un tópico espurio.

La **NPMI de coherencia** (ventana deslizante de 10 sobre el held-out) mide otra cosa que la NPMI de colocación usada para elegir bigramas: la primera mide si las palabras top de un tópico co-ocurren en texto no visto; la segunda, si dos lemas van pegados.

In [ ]:
t0 = time.time()
bg = ag.tabla_bigramas(dfp)
LB = ag.leer_lista_blanca(utils.data_dir() / "lista_blanca_con_propn.txt")
cp = set(bg["bigrama"])
print(f"Tabla NPMI: {len(bg):,} candidatos | {time.time() - t0:.0f} s")
print(
    f"Lista blanca: {len(LB)} | presentes en la tabla: {len(set(LB) & cp)} | faltan: {sorted(set(LB) - cp)}"
)
print(
    bg[bg.bigrama.isin(LB)][["bigrama", "n", "df", "npmi", "tipo"]]
    .sort_values("df", ascending=False)
    .round(3)
    .to_string(index=False)
)
EST = ag.estrategias_bigramas(bg, LB, cfg)
print({k: len(v) for k, v in EST.items()})
PROHIBIDAS = {
    w
    for s, ws in utils.leer_stopwords_dominio().items()
    if s in ("dias", "meses", "nombres_pila", "metatexto", "scraping")
    for w in ws
}

In [ ]:
def representar(bigr, idx_todos=None):
    kw = dict(bigramas=bigr, **KW)
    tok_fit = ag.tokens_corpus(dfp, idx_fit, **kw)
    vec, X_fit = ag.vectorizar(tok_fit, cfg)
    ho_full = ag.tokens_corpus(dfp, idx_ho, completo=True, **kw)
    X_ho = vec.transform([[t for t in d if t] for d in ho_full])
    feat = vec.get_feature_names_out()
    dfr = np.asarray((X_fit > 0).sum(0)).ravel() / X_fit.shape[0]
    rep = dict(
        bigr=bigr,
        vec=vec,
        X_fit=X_fit,
        X_ho=X_ho,
        ho_full=ho_full,
        feat=feat,
        df_ratio=dict(zip(feat, dfr)),
    )
    if idx_todos is not None:
        rep["X_all"] = vec.transform(ag.tokens_corpus(dfp, idx_todos, **kw))
    return rep


def evaluar(rep, fits, cv=True):
    # Métricas por (K, semilla) + estabilidad por K. Las co-ocurrencias se cuentan una vez para todos los modelos.
    feat = rep["feat"]
    union = {
        w
        for r in fits.values()
        for t in ag.palabras_top(r["H"], feat, cfg.topn_coh)
        for w in t
    }
    coh = ag.Coherencia(rep["ho_full"], sorted(union), cfg.ventana_npmi)
    coh_cv = (
        ag.Coherencia(rep["ho_full"], sorted(union), cfg.ventana_cv) if cv else None
    )
    filas, npmi_t = [], {}
    for (K, s), r in fits.items():
        m = ag.metricas_modelo(r, feat, coh, cfg, coh_cv, rep["X_fit"], rep["X_ho"])
        npmi_t[(K, s)] = m.pop("_npmi_topicos")
        filas.append(m)
    tab = pd.DataFrame(filas)
    extra = []
    for K in sorted({k for k, _ in fits}):
        Hs = {s: fits[(K, s)]["H"] for (k, s) in fits if k == K}
        med = ag.medoide(Hs)
        est_t = ag.estabilidad_por_topico(
            Hs[med], [h for s, h in Hs.items() if s != med]
        )
        res = ag.marcar_residuales(
            Hs[med], feat, npmi_t[(K, med)], est_t, rep["df_ratio"], PROHIBIDAS, cfg
        )
        extra.append(
            {
                "K": K,
                **ag.estabilidad(list(Hs.values()), cfg.topn_coh),
                "medoide": med,
                "pct_residuales": res.residual_sugerido.mean() * 100,
            }
        )
    return tab.merge(pd.DataFrame(extra), on="K"), npmi_t, coh

In [ ]:
# 💰 10 representaciones × 3 semillas con K = k_ref
t0 = time.time()
filas_b, top_bi = [], []
for nombre, bigr in EST.items():
    rep = representar(bigr)
    fits = ag.rejilla_nmf(rep["X_fit"], [cfg.k_ref], cfg.semillas, cfg)
    tab_b, _, _ = evaluar(rep, fits, cv=False)
    tab_b.insert(0, "estrategia", nombre)
    tab_b["n_bigramas"] = len(bigr)
    tab_b["vocab"] = len(rep["feat"])
    filas_b.append(tab_b)
    H = fits[(cfg.k_ref, int(tab_b.medoide.iloc[0]))]["H"]
    for k, t in enumerate(ag.palabras_top(H, rep["feat"], 10)):
        bi = [w for w in t if "_" in w and w != "estados_unidos"]
        top_bi.append(
            {
                "estrategia": nombre,
                "topico": f"T{k:02d}",
                "bigramas_top10": " ".join(bi),
                "n": len(bi),
            }
        )
    print(
        f"{nombre:6s} | {len(bigr):5d} bigramas | vocab {len(rep['feat']):,} | NPMI {tab_b.npmi_media.mean():.4f} | {time.time() - t0:.0f} s"
    )
    del rep, fits
TIEMPOS["comparacion_bigramas_s"] = round(time.time() - t0)
CB = pd.concat(filas_b, ignore_index=True)

In [ ]:
agg_b = ag.regla_1se(CB, "estrategia")
TB = pd.DataFrame(top_bi)
agg_b["bigramas_en_top10"] = TB.groupby("estrategia")["n"].sum()
ESTRATEGIA, motivo_b = ag.elegir_estrategia(agg_b)
cols_b = [
    "npmi_media",
    "se",
    "npmi_min",
    "diversidad",
    "estab_cos",
    "estab_jaccard",
    "pct_residuales",
    "err_rel_heldout",
    "n_bigramas",
    "vocab",
    "bigramas_en_top10",
]
print(agg_b[cols_b].sort_values("npmi_media", ascending=False).round(4).to_string())
print(f"\n→ Estrategia elegida: {ESTRATEGIA} | {motivo_b}")
agg_b.round(5).to_csv(OUT / "comparacion_bigramas.csv")
CB.round(5).to_csv(OUT / "comparacion_bigramas_por_semilla.csv", index=False)
TB.to_csv(OUT / "comparacion_bigramas_top10.csv", index=False)
print("\nBigramas en el top-10 de los tópicos (semilla medoide):")
print(
    TB[TB.n > 0]
    .groupby("estrategia")["bigramas_top10"]
    .apply(lambda s: " | ".join(s))
    .to_string()
)

**Lectura.** La tabla de arriba es la evidencia de la elección. Que una estrategia meta bigramas al top-10 no la hace mejor: cuenta solo si sube la coherencia held-out sin bajar la estabilidad ni subir los residuales. Si la tabla no muestra mejora sobre A, el modelo va sin bigramas y así se reporta. La estrategia elegida es la que se usa en la rejilla completa de K.

## 4. NMF y selección de K
- **Rejilla:** K de 8 a 20, 3 semillas por K, todas con `init="random"` y distinto `random_state`. Con `nndsvd`/`nndsvda`, que son deterministas, las "semillas" darían el mismo ajuste y la estabilidad saldría ≈1 de forma artificial. Ese fue uno de los problemas del 06, que comparaba `nndsvda` contra `random`.
- **Métricas por K:**
  - **C_NPMI held-out** (Röder et al. 2015): ventana deslizante de 10 posiciones sobre `lemas_contenido` ya unido (las posiciones de verbos y nombres propios cuentan para la distancia), top-10, media y mínimo por tópico;
  - **C_v**, como segunda métrica de coherencia: ventana 110, mismas co-ocurrencias y costo marginal. No entra en la regla, porque se ha documentado que es inestable y a veces premia tópicos malos;
  - **diversidad:** palabras únicas en el top-25 / (K·25);
  - **estabilidad:** emparejamiento húngaro por coseno de H y Jaccard del top-10;
  - **error de reconstrucción relativo** ‖X−WH‖/‖X‖, en ajuste y en held-out.
- **Sin perplejidad:** NMF minimiza una pérdida de Frobenius y no es un modelo generativo, así que no define una verosimilitud de los documentos. La perplejidad no existe para NMF. Incluso en LDA se ha mostrado que correlaciona mal con el juicio humano de coherencia (Chang et al. 2009).

In [ ]:
# 💰 39 ajustes en paralelo + proyección de los 91k documentos
t0 = time.time()
REP = representar(EST[ESTRATEGIA], idx_todos=np.arange(len(dfp)))
FEAT = REP["feat"]
print(
    f"Representación {ESTRATEGIA}: vocab {len(FEAT):,} | X_fit {REP['X_fit'].shape} | X_all {REP['X_all'].shape} | {time.time() - t0:.0f} s"
)
FITS = ag.rejilla_nmf(REP["X_fit"], cfg.ks, cfg.semillas, cfg)
TIEMPOS["rejilla_K_s"] = round(time.time() - t0)
MK, NPMI_T, COH = evaluar(REP, FITS, cv=True)
TIEMPOS["rejilla_K_metricas_s"] = round(time.time() - t0)
print(f"Rejilla + métricas: {TIEMPOS['rejilla_K_metricas_s']} s")

In [ ]:
agg_k = ag.regla_1se(MK, "K")
K_STAR, motivo_k = ag.elegir_K(agg_k, cfg)
cols_k = [
    "npmi_media",
    "se",
    "npmi_min",
    "npmi_mediana",
    "cv_media",
    "diversidad",
    "estab_cos",
    "estab_jaccard",
    "err_rel_ajuste",
    "err_rel_heldout",
    "pct_residuales",
    "n_iter",
    "convergio",
]
print(agg_k[cols_k].round(4).to_string())
print(f"\n→ K* = {K_STAR} | {motivo_k}")
agg_k.round(5).to_csv(OUT / "metricas_K.csv")
MK.round(5).to_csv(OUT / "metricas_K_por_semilla.csv", index=False)

fig_, ax = plt.subplots(2, 3, figsize=(15, 7))
ax = ax.ravel()
x = agg_k.index
ax[0].errorbar(x, agg_k.npmi_media, yerr=agg_k.se, fmt="o-")
ax[0].set_title("C_NPMI held-out (media ± EE)")
ax[1].plot(x, agg_k.npmi_min, "o-")
ax[1].set_title("C_NPMI mínimo por tópico")
ax[2].plot(x, agg_k.cv_media, "o-")
ax[2].set_title("C_v (secundaria)")
ax[3].plot(x, agg_k.diversidad, "o-")
ax[3].set_title("Diversidad top-25")
ax[4].plot(x, agg_k.estab_cos, "o-", label="coseno")
ax[4].plot(x, agg_k.estab_jaccard, "s--", label="Jaccard top-10")
ax[4].axhline(cfg.estab_min, color="k", ls=":")
ax[4].legend()
ax[4].set_title("Estabilidad entre semillas")
ax[5].plot(x, agg_k.err_rel_ajuste, "o-", label="ajuste")
ax[5].plot(x, agg_k.err_rel_heldout, "s--", label="held-out")
ax[5].legend()
ax[5].set_title("Error relativo de reconstrucción")
for a in ax:
    a.axvline(K_STAR, color="r", alpha=0.4)
    a.set_xlabel("K")
fig("fig_metricas_K.png")

**Cómo leer las métricas.**
- La **coherencia** dice si las palabras top de cada tópico aparecen juntas en texto no visto. En noticias, valores de C_NPMI cercanos a 0 son pobres y de 0,1 o más, buenos. El **mínimo por tópico** importa más que la media para la observación (a).
- La **estabilidad** dice si el tópico es del corpus o del azar de la inicialización.
- La **diversidad** baja cuando los tópicos repiten palabras, que es la señal de un K excesivo (el problema de K = 40).
- El **error de reconstrucción** siempre baja con K, así que no sirve solo para elegir. Una brecha creciente entre ajuste y held-out indica sobreajuste.

**Si K\* cae en el borde de la rejilla.** El EE entre semillas mide el ruido de optimización, no diferencias sustantivas entre K. Cuando es muy chico, la regla de 1 EE casi equivale a quedarse con el máximo, y la C_NPMI tiende a crecer con K porque los tópicos se vuelven más específicos. La regla se mantiene tal como se declaró (no se cambia después de ver los resultados). Como control, la sección 7 reporta los cambios con K = 10, 12, 14 y 16 como series informativas: si los cambios de consenso aparecen también ahí, la conclusión no depende de K.

Override manual (solo a un K adyacente y con el motivo escrito):

In [ ]:
K_MANUAL, MOTIVO_MANUAL = None, ""
if K_MANUAL is not None:
    assert abs(K_MANUAL - K_STAR) <= 1 and MOTIVO_MANUAL, (
        "solo K adyacente y con motivo"
    )
    K_STAR = K_MANUAL
    motivo_k += f" | override manual: {MOTIVO_MANUAL}"
SEM = {s: FITS[(K_STAR, s)] for s in cfg.semillas}
MED = ag.medoide({s: r["H"] for s, r in SEM.items()})
MODELO, H = SEM[MED]["modelo"], SEM[MED]["H"]
print(f"K* = {K_STAR} | semilla medoide = {MED}")

## 5. Calidad de tópicos (K elegido)
θ se proyecta a los 91.585 documentos con el H fijo del ajuste. Los titulares representativos se toman entre los **no breves**: en las breves un solo tópico suele llevarse todo θ por tener pocas palabras, y sus titulares no representan el tema.

In [ ]:
# 💰 proyección
t0 = time.time()
T_ALL, OK = ag.theta(MODELO.transform(REP["X_all"]), H)
TIEMPOS["proyeccion_s"] = round(time.time() - t0)
print(
    f"θ: {T_ALL.shape} | docs sin ningún tópico: {(~OK).sum():,} ({(~OK).mean():.2%}) | {TIEMPOS['proyeccion_s']} s"
)
print(
    "Vacíos por estrato:",
    pd.crosstab(fuentes[~OK], breve[~OK]).rename(index=ag.CORTO).to_dict(),
)

est_t = ag.estabilidad_por_topico(H, [SEM[s]["H"] for s in cfg.semillas if s != MED])
tops10 = ag.palabras_top(H, FEAT, 10)
tops15 = ag.palabras_top(H, FEAT, 15)
coh_cv = ag.Coherencia(
    REP["ho_full"], sorted({w for t in tops10 for w in t}), cfg.ventana_cv
)
RES = ag.marcar_residuales(
    H,
    FEAT,
    NPMI_T[(K_STAR, MED)],
    est_t,
    REP["df_ratio"],
    PROHIBIDAS,
    cfg,
    T=T_ALL[OK],
    es_breve=breve[OK],
    log_long=np.log1p(dfp["n_palabras_cuerpo"].values[OK]),
)
prev_f = (
    pd.DataFrame(T_ALL[OK])
    .groupby(fuentes[OK])
    .mean()
    .T.rename(columns=lambda c: "prev_" + ag.CORTO[c])
)
TOP = pd.DataFrame(
    {
        "topico": [f"T{k:02d}" for k in range(K_STAR)],
        "top10": [" ".join(t) for t in tops10],
        "top15": [" ".join(t) for t in tops15],
        "npmi": NPMI_T[(K_STAR, MED)].round(4),
        "cv": [round(coh_cv.cv(t), 4) for t in tops10],
        "estabilidad": est_t.round(3),
        "prev_global": T_ALL[OK].mean(0).round(4),
    }
)
TOP = pd.concat([TOP, prev_f.round(4).reset_index(drop=True)], axis=1).merge(
    RES, on="topico"
)
NOM, estado = ag.tabla_nombres(OUT / "nombres_topicos.csv", TOP)
print("nombres_topicos.csv:", estado)
TOP = TOP.merge(NOM[["topico", "nombre", "residual"]], on="topico")
RESID = {k for k, r in enumerate(TOP["residual"]) if r}
NOMBRE = dict(zip(range(K_STAR), TOP["topico"] + " " + TOP["nombre"]))
print(
    TOP[
        [
            "topico",
            "npmi",
            "cv",
            "estabilidad",
            "prev_global",
            "prev_eltiempo",
            "prev_semana",
            "prev_dinero",
            "residual",
            "motivo",
            "top15",
        ]
    ].to_string(index=False)
)

In [ ]:
nb = ~breve & OK
TIT = ag.titulares_representativos(
    T_ALL[nb], dfp["titulo"].values[nb], fuentes[nb], dfp["fecha"].values[nb], 5
)
for k in range(K_STAR):
    print(
        f"\n{NOMBRE[k]}  {'[RESIDUAL: ' + TOP.motivo[k] + ']' if k in RESID else ''}\n  {TOP.top15[k]}"
    )
    for t in TIT[k]:
        print("   ", t)

**Revisión manual.** Edita `results/08/nombres_topicos.csv`: pon un `nombre` corto y marca o desmarca `residual` (True/False). Después vuelve a correr desde la celda anterior; si el modelo no cambió, tus ediciones se respetan. Los residuales se excluyen de la interpretación y de la agenda (θ se renormaliza sobre los demás), pero siguen en `topicos.csv` con su motivo, y la robustez incluye la serie "con residuales" como información.

In [ ]:
print(f"Residuales: {sorted(NOMBRE[k] for k in RESID)}")
T_AG, OK_AG, KEEP = ag.theta_agenda(T_ALL, OK, RESID)
assert len(KEEP) >= 2, (
    "casi todos los tópicos quedaron residuales: revisa nombres_topicos.csv y los umbrales de residual"
)
NOMBRES_AG = [NOMBRE[k] for k in KEEP]
print(f"Tópicos de agenda: {len(KEEP)} | docs con masa en ellos: {OK_AG.sum():,}")
TOP.to_csv(OUT / "topicos.csv", index=False)

## 6. Agenda y estabilidad
### 6.1 Prevalencias e intervalos bootstrap
Se usan cuatro series de El Tiempo (1990–2011): {todos los docs, sin breves} × {media por documento, igual peso por mes válido}. Los IC al 95 % salen de un bootstrap percentil que remuestrea documentos dentro del año, o dentro de cada mes válido en la serie por mes, con B = 1000.

In [ ]:
DESDE = {
    ag.ET: cfg.principal_desde,
    ag.SEMANA: cfg.año_min,
    ag.DINERO: int(años[fuentes == ag.DINERO].min()),
}


def mascara(fuente, sin_breves, ok=None):
    ok = OK_AG if ok is None else ok
    m = ok & (años >= DESDE.get(fuente, cfg.año_min))
    if fuente:
        m &= fuentes == fuente
    if sin_breves:
        m &= ~breve
    return m


def serie(T, fuente, sin_breves, modo, ok=None):
    m = mascara(fuente, sin_breves, ok)
    rango = np.arange(DESDE.get(fuente, cfg.año_min), cfg.año_max + 1)
    if modo == "doc":
        P, n = ag.prevalencia(T[m], años[m], años_rango=rango)
    else:
        P, n = ag.prevalencia(
            T[m], años[m], meses[m], VALIDO[fuente], "mes", rango, cfg.min_meses_validos
        )
    return P, n, m


SERIES = {}
for f in [ag.ET, ag.SEMANA, ag.DINERO]:
    for sb in (False, True):
        for modo in ("doc", "mes"):
            SERIES[(ag.CORTO[f], sb, modo)] = serie(T_AG, f, sb, modo)
print(
    {k: int((v[1] >= cfg.min_n_año).sum()) for k, v in SERIES.items()},
    "← años con N suficiente",
)

In [ ]:
# 💰 bootstrap
t0 = time.time()
prev_ic = []
for (f, sb, modo), (P, n, m) in SERIES.items():
    fuente = {v: k for k, v in ag.CORTO.items()}[f]
    ic = ag.bootstrap_prevalencia(
        T_AG[m],
        años[m],
        meses[m],
        VALIDO[fuente],
        modo,
        cfg.b_boot,
        cfg.seed,
        P.index.values,
        cfg.min_meses_validos,
    )
    pl = pd.DataFrame(
        [
            (int(y), k, P.loc[y, k])
            for y in P.index
            for k in P.columns
            if pd.notna(P.loc[y, k])
        ],
        columns=["año", "topico", "prev"],
    )
    if pl.empty or ic.empty:
        continue
    d = pl.merge(ic, on=["año", "topico"], how="left").merge(
        n.rename("n").rename_axis("año").reset_index(), on="año"
    )
    d.insert(0, "serie", f"{f}|{'sin_breves' if sb else 'todos'}|{modo}")
    prev_ic.append(d)
PREV_IC = pd.concat(prev_ic, ignore_index=True)
PREV_IC["topico"] = PREV_IC["topico"].map(dict(enumerate(NOMBRES_AG)))
PREV_IC.round(5).to_csv(OUT / "prevalencias_ic.csv", index=False)
TIEMPOS["bootstrap_s"] = round(time.time() - t0)
print(f"{TIEMPOS['bootstrap_s']} s | {len(PREV_IC):,} filas")

In [ ]:
ncol = 4
nrow = int(np.ceil(len(KEEP) / ncol))
fig_, axs = plt.subplots(nrow, ncol, figsize=(16, 2.6 * nrow), sharex=True)
axs = np.ravel(axs)
for j, nom in enumerate(NOMBRES_AG):
    for s, est in [("eltiempo|todos|doc", "C0-"), ("eltiempo|sin_breves|mes", "C3-")]:
        d = PREV_IC[
            (PREV_IC.serie == s)
            & (PREV_IC.topico == nom)
            & (PREV_IC.n >= cfg.min_n_año)
        ]
        axs[j].plot(d.año, d.prev, est, lw=1.2, label=s.replace("eltiempo|", ""))
        axs[j].fill_between(d.año, d.lo, d.hi, color=est[:2], alpha=0.2)
    axs[j].set_title(nom[:40], fontsize=8)
for a in axs[len(KEEP) :]:
    a.axis("off")
axs[0].legend(fontsize=7)
fig("fig_prevalencia_ic_eltiempo.png")

### 6.2 Transiciones año a año contra un nulo estratificado
El nulo permuta las etiquetas de año **dentro de cada estrato fuente × es_breve**. Así, que 2000–2004 tenga más breves que 2006 no puede producir divergencia por sí solo. Una variante permuta además dentro del **mes**, lo que neutraliza los huecos de cobertura mensual. Se reportan dos tamaños de efecto:
- el **exceso** = JS observado − media del nulo;
- la **fracción de la distancia entre fuentes** = exceso / exceso medio de El Tiempo vs. Semana en el mismo año.

In [ ]:
# 💰 nulos por permutación
t0 = time.time()
EST_FB = np.array([f"{a}|{b}" for a, b in zip(fuentes, breve)])
EST_FBM = np.array([f"{a}|{b}|{c}" for a, b, c in zip(fuentes, breve, meses)])
R = lambda f: np.arange(DESDE.get(f, cfg.año_min), cfg.año_max + 1)
TR = {
    "eltiempo | estrato fuente×breve": ag.transiciones(
        T_AG,
        años,
        EST_FB,
        mascara(ag.ET, False),
        R(ag.ET),
        B=cfg.b_perm,
        min_n=cfg.min_n_año,
    ),
    "eltiempo | estrato fuente×breve×mes": ag.transiciones(
        T_AG,
        años,
        EST_FBM,
        mascara(ag.ET, False),
        R(ag.ET),
        B=cfg.b_perm,
        min_n=cfg.min_n_año,
    ),
    "eltiempo sin breves | estrato fuente×mes": ag.transiciones(
        T_AG,
        años,
        EST_FBM,
        mascara(ag.ET, True),
        R(ag.ET),
        B=cfg.b_perm,
        min_n=cfg.min_n_año,
    ),
    "semana | estrato fuente×breve": ag.transiciones(
        T_AG,
        años,
        EST_FB,
        mascara(ag.SEMANA, False),
        R(ag.SEMANA),
        B=cfg.b_perm,
        min_n=cfg.min_n_año,
    ),
    "dinero | estrato fuente×breve": ag.transiciones(
        T_AG,
        años,
        EST_FB,
        mascara(ag.DINERO, False),
        R(ag.DINERO),
        B=cfg.b_perm,
        min_n=cfg.min_n_año,
    ),
    "todas | estrato fuente×breve": ag.transiciones(
        T_AG,
        años,
        EST_FB,
        mascara(None, False),
        np.arange(cfg.año_min, cfg.año_max + 1),
        B=cfg.b_perm,
        min_n=cfg.min_n_año,
    ),
}
REF = ag.js_entre_fuentes(
    T_AG,
    años,
    fuentes,
    breve,
    mascara(None, False),
    ag.ET,
    ag.SEMANA,
    R(ag.ET),
    B=cfg.b_perm,
    min_n=cfg.min_n_año,
)
TIEMPOS["transiciones_s"] = round(time.time() - t0)
REF_EXC = REF["exceso"].mean()
print(
    f"{TIEMPOS['transiciones_s']} s | JS El Tiempo vs Semana (mismo año): {REF.js.mean():.4f}, ruido {REF.nulo_media.mean():.4f}, exceso {REF_EXC:.4f}"
)
res_tr = []
for k, r in TR.items():
    if not len(r):
        continue
    r["frac_dist_fuentes"] = r["exceso"] / REF_EXC
    res_tr.append(
        {
            "serie": k,
            "transiciones": len(r),
            "js_medio": r.js.mean(),
            "ruido_medio": r.nulo_media.mean(),
            "exceso_medio": r.exceso.mean(),
            "exceso_max": r.exceso.max(),
            "año_exceso_max": int(r.loc[r.exceso.idxmax(), "año2"]),
            "frac_dist_fuentes_media": r.frac_dist_fuentes.mean(),
            "anexo_pct_BH5": (r.p_bh < 0.05).mean() * 100,
        }
    )
RES_TR = pd.DataFrame(res_tr)
print(RES_TR.round(4).to_string(index=False))
pd.concat([r.assign(serie=k) for k, r in TR.items() if len(r)]).round(5).to_csv(
    OUT / "transiciones.csv", index=False
)
REF.round(5).to_csv(OUT / "js_entre_fuentes.csv", index=False)
print("\nEl Tiempo, transiciones con mayor exceso:")
print(
    TR["eltiempo | estrato fuente×breve"]
    .sort_values("exceso", ascending=False)
    .head(8)[
        ["año", "año2", "n1", "n2", "js", "nulo_media", "exceso", "frac_dist_fuentes"]
    ]
    .round(4)
    .to_string(index=False)
)

In [ ]:
claves = list(TR)[:4]
fig_, axs = plt.subplots(1, len(claves), figsize=(18, 3.8))
for ax, k in zip(axs, claves):
    r = TR[k]
    if not len(r):
        continue
    ax.plot(r.año2, r.js, "o-", label="JS observado")
    ax.plot(r.año2, r.nulo_media, "k--", label="ruido")
    ax.fill_between(r.año2, r.nulo_media, r.nulo_p95, color="gray", alpha=0.3)
    ax.set_title(k, fontsize=9)
    ax.axhline(REF.js.mean(), color="C3", ls=":", label="JS ET vs Semana")
axs[0].legend(fontsize=7)
fig("fig_js_vs_nulo.png")

### 6.3 Curva de decaimiento por rezago

In [ ]:
fig_, ax = plt.subplots(figsize=(8, 4.5))
LAG = []
for clave, et in [
    (("eltiempo", False, "doc"), "eltiempo todos/doc"),
    (("eltiempo", True, "mes"), "eltiempo sin breves/mes"),
    (("semana", False, "doc"), "semana todos/doc"),
    (("dinero", False, "doc"), "dinero todos/doc"),
]:
    P, n, _ = SERIES[clave]
    P = P.copy()
    P[(n < cfg.min_n_año).values] = np.nan
    l = ag.curva_rezago(P, cfg.lag_max).assign(serie=et)
    LAG.append(l)
    if l.js_medio.notna().sum() >= 3:
        ax.plot(l.rezago, l.js_medio, "o-", ms=3, label=et)
ax.axhline(
    TR["eltiempo | estrato fuente×breve"].nulo_media.mean(),
    color="k",
    ls=":",
    label="ruido (El Tiempo)",
)
ax.axhline(REF.js.mean(), color="C3", ls=":", label="JS entre fuentes")
ax.set_xlabel("rezago (años)")
ax.set_ylabel("JS medio")
ax.legend(fontsize=8)
fig("fig_decaimiento_rezago.png")
LAG = pd.concat(LAG)
LAG.round(5).to_csv(OUT / "decaimiento_rezago.csv", index=False)
print(
    LAG.pivot(index="rezago", columns="serie", values="js_medio").round(4).to_string()
)

**Cómo leer esto.**
- Si el exceso a rezago 1 es pequeño frente a la distancia entre fuentes, la agenda año a año es estable: cambia menos de un año a otro que de un periódico a otro.
- Si la curva crece con el rezago sin saturarse, hay **deriva gradual**. Si crece y se aplana, hay **regímenes**.
- Los puntos de cambio (sección 7) solo tienen sentido en el segundo caso, o como escalones de una deriva.

## 7. Puntos de cambio
Se usa el mismo `binseg` del 06: raíz de la prevalencia (≈ Hellinger), peso min(n, 3000) y umbral de cada división por permutación del orden de los años (percentil 95 de 200 permutaciones). Cada cambio es el **primer año del nuevo régimen**.

**Prueba clave de artefacto:** el mismo detector corre sobre las **series de formato** de El Tiempo: % breves, log de la mediana de palabras, log de docs/año y dos marcadores del régimen de archivo que midió el 07 (% de titulares en Mayúscula Inicial y % de lead duplicado). Cada serie se evalúa sola y todas juntas. Para cada cambio temático se indica si coincide (±1 año) con un cambio de formato y si sobrevive a los controles.

In [ ]:
t0 = time.time()
DET = []


def det(clave, nombre):
    P, n, _ = SERIES[clave]
    d = ag.detectar(P, n, nombre, cfg)
    DET.append(d)
    return d["cambios"]


S4 = {
    "todos_doc": det(("eltiempo", False, "doc"), "ET todos | media doc"),
    "sin_breves_doc": det(("eltiempo", True, "doc"), "ET sin breves | media doc"),
    "todos_mes": det(("eltiempo", False, "mes"), "ET todos | peso por mes"),
    "sin_breves_mes": det(("eltiempo", True, "mes"), "ET sin breves | peso por mes"),
}
CONTRASTE = {
    "semana_todos_doc": det(("semana", False, "doc"), "Semana todos | media doc"),
    "semana_sin_breves_mes": det(
        ("semana", True, "mes"), "Semana sin breves | peso por mes"
    ),
    "dinero_todos_doc": det(("dinero", False, "doc"), "Dinero todos | media doc"),
}
fmt_et = fmt[(fmt.fuente == ag.ET) & (fmt.año >= cfg.principal_desde)].set_index("año")
DET_FMT = ag.detectar_formato(fmt_et, "formato ET", cfg)
FORMATO = {d["variante"]: d["cambios"] for d in DET_FMT}
fmt_se = fmt[fmt.fuente == ag.SEMANA].set_index("año")
DET_FMT_SE = ag.detectar_formato(fmt_se, "formato Semana", cfg)
for d in DET + DET_FMT + DET_FMT_SE:
    print(
        f"{d['variante']:55s} | {d['años_validos']:16s} | {d['cambios']} fuerza {d['fuerza']}"
    )

### 7.1 Variantes votantes
Las variantes se evalúan sobre la serie con **ambos controles** (sin breves y peso por mes), que es la que decide el consenso:
- las otras 2 semillas;
- K−2 y K+2 (semilla medoide de cada K);
- con/sin bigramas;
- con/sin lista blanca.

Los residuales se trasladan a cada variante emparejando sus tópicos con los del modelo elegido (coseno ≥ 0,5 sobre el vocabulario común).

In [ ]:
# 💰 variantes (cada representación nueva reconstruye tokens y proyecta 91k docs)
t0 = time.time()
VOT = {}


def serie_variante(H_v, W_all_v, feat_v, nombre, destino=None):
    T_v, ok_v = ag.theta(W_all_v, H_v)
    res_v = ag.residuales_por_emparejamiento(H, FEAT, RESID, H_v, feat_v)
    T_va, ok_va, _ = ag.theta_agenda(T_v, ok_v, res_v)
    P, n, _ = serie(T_va, ag.ET, True, "mes", ok=ok_va)
    d = ag.detectar(P, n, nombre, cfg)
    DET.append(d)
    (VOT if destino is None else destino)[nombre] = d["cambios"]
    print(f"{nombre:28s} | residuales {len(res_v)} | cambios {d['cambios']}")


for s in cfg.semillas:
    if s != MED:
        serie_variante(
            SEM[s]["H"], SEM[s]["modelo"].transform(REP["X_all"]), FEAT, f"semilla {s}"
        )
for K in (K_STAR - 2, K_STAR + 2):
    fk = (
        {s: FITS[(K, s)] for s in cfg.semillas}
        if (K, cfg.semillas[0]) in FITS
        else {
            r["seed"]: r
            for r in ag.rejilla_nmf(REP["X_fit"], [K], cfg.semillas, cfg).values()
        }
    )
    mk = ag.medoide({s: r["H"] for s, r in fk.items()})
    serie_variante(
        fk[mk]["H"], fk[mk]["modelo"].transform(REP["X_all"]), FEAT, f"K={K}"
    )

base = ESTRATEGIA.replace("+LB", "")
alt_bi = (
    "A"
    if base != "A"
    else agg_b.drop(index=["A", "A+LB"]).npmi_media.idxmax().replace("+LB", "")
) + ("+LB" if "+LB" in ESTRATEGIA else "")
alt_lb = ESTRATEGIA.replace("+LB", "") if "+LB" in ESTRATEGIA else ESTRATEGIA + "+LB"
for alt, nombre in [
    (alt_bi, f"bigramas: {alt_bi}"),
    (alt_lb, f"lista blanca: {alt_lb}"),
]:
    rep_v = representar(EST[alt], idx_todos=np.arange(len(dfp)))
    fv = {
        r["seed"]: r
        for r in ag.rejilla_nmf(rep_v["X_fit"], [K_STAR], cfg.semillas, cfg).values()
    }
    mv_ = ag.medoide({s: r["H"] for s, r in fv.items()})
    serie_variante(
        fv[mv_]["H"], fv[mv_]["modelo"].transform(rep_v["X_all"]), rep_v["feat"], nombre
    )
    del rep_v
TIEMPOS["variantes_s"] = round(time.time() - t0)

# informativas (no votan): con residuales y sensibilidad a K en la franja 10–16
P, n, _ = serie(T_ALL, ag.ET, True, "mes", ok=OK)
d = ag.detectar(P, n, "con residuales (sin breves, mes)", cfg)
DET.append(d)
INFO = {"con_residuales": d["cambios"]}
for K in (10, 12, 14, 16):
    if K in (K_STAR, K_STAR - 2, K_STAR + 2) or (K, cfg.semillas[0]) not in FITS:
        continue
    fk = {s: FITS[(K, s)] for s in cfg.semillas}
    mk = ag.medoide({s: r["H"] for s, r in fk.items()})
    serie_variante(
        fk[mk]["H"],
        fk[mk]["modelo"].transform(REP["X_all"]),
        FEAT,
        f"K={K}",
        destino=INFO,
    )
print(f"variantes: {TIEMPOS['variantes_s']} s")

In [ ]:
ROB = ag.tabla_robustez(S4, VOT, FORMATO, CONTRASTE, INFO, cfg)
CONSENSO = ROB.loc[ROB.veredicto.str.startswith("robusto"), "año"].astype(int).tolist()
print(ROB.to_string(index=False))
print("\nConsenso (robustos):", CONSENSO)
ROB.to_csv(OUT / "robustez_cambios.csv", index=False)
pd.DataFrame(
    [
        {
            "variante": d["variante"],
            "años_validos": d["años_validos"],
            "cambios": ", ".join(map(str, d["cambios"])) or "—",
            "fuerza": ", ".join(map(str, d["fuerza"])),
        }
        for d in DET + DET_FMT + DET_FMT_SE
    ]
).to_csv(OUT / "cambios_detectados.csv", index=False)
print("\nCambios de la entrega 2 (1995, 1999/2000, 2006):")
for c in (1995, 1999, 2000, 2006):
    r = ROB[(ROB.año - c).abs() <= cfg.ventana_coincidencia]
    print(
        f"  {c}: "
        + (
            "; ".join(
                f"{int(x.año)} → {x.veredicto} (formato: {x.series_formato or 'no'})"
                for x in r.itertuples()
            )
            if len(r)
            else "no detectado en ninguna serie de control"
        )
    )

In [ ]:
P, n, _ = SERIES[("eltiempo", True, "mes")]
P = P.copy()
P[(n < cfg.min_n_año).values] = np.nan
fig_, ax = plt.subplots(
    2, 1, figsize=(13, 8), gridspec_kw={"height_ratios": [2.2, 1]}, sharex=True
)
yrs = P.index.values
im = ax[0].imshow(
    (P / P.mean()).T.values,
    aspect="auto",
    cmap="RdBu_r",
    vmin=0,
    vmax=2,
    extent=[yrs[0] - 0.5, yrs[-1] + 0.5, len(KEEP) - 0.5, -0.5],
)
ax[0].set_yticks(range(len(KEEP)))
ax[0].set_yticklabels([s[:35] for s in NOMBRES_AG], fontsize=7)
for c in CONSENSO:
    ax[0].axvline(c - 0.5, color="k", lw=2)
ax[0].set_title(
    "El Tiempo sin breves, peso por mes: prevalencia relativa (1 = media) y cambios de consenso"
)
Z = fmt_et[["pct_breve", "mediana_palabras", "docs"]]
Z = (Z - Z.mean()) / Z.std()
for c in Z:
    ax[1].plot(Z.index, Z[c], "o-", ms=3, label=c)
for c in sorted({a for v in FORMATO.values() for a in v}):
    ax[1].axvline(c - 0.5, color="gray", ls="--")
for c in CONSENSO:
    ax[1].axvline(c - 0.5, color="k", lw=2)
ax[1].legend(fontsize=7)
ax[1].set_title(
    "Series de formato (z) y sus cambios (gris) vs. cambios temáticos (negro)"
)
fig("fig_cambios_formato.png")

## 8. Contraste histórico
Se usa la lista `EVENTOS` del 06 **tal cual**, congelada. Se cuentan los cambios de consenso que caen a ±1 año de un evento y se comparan con la **línea base aleatoria**: el mismo número de cambios en años al azar, elegidos entre los años en que el detector puede marcar un inicio de régimen.

In [ ]:
# LISTA CONGELADA (copiada del 06, no editar)
EVENTOS = {
    1985: "Toma del Palacio de Justicia / Armero",
    1989: "Asesinato de Galán; narcoterrorismo",
    1991: "Constitución de 1991 / apertura económica",
    1993: "Muerte de Pablo Escobar",
    1996: "Proceso 8000",
    1998: "Pastrana, zona de distensión del Caguán",
    2002: "Uribe y 'seguridad democrática'",
    2006: "Reelección de Uribe / parapolítica",
    2010: "Santos / ola invernal",
}
P, n, _ = SERIES[("eltiempo", True, "mes")]
VALID_Y = [
    int(y) for y in P.index if n[y] >= cfg.min_n_año and not P.loc[y].isna().any()
]
LBE = ag.linea_base_eventos(
    CONSENSO, EVENTOS, VALID_Y, seed=cfg.seed, min_seg=cfg.min_seg
)
for c in CONSENSO:
    print(
        f"  {c} → {[f'{e}: {t}' for e, t in EVENTOS.items() if abs(c - e) <= 1] or 'ninguno'}"
    )
print(LBE)

## 9. Exportes

In [ ]:
th = pd.DataFrame(T_ALL.round(5), columns=[f"T{k:02d}" for k in range(K_STAR)])
th.insert(0, "id", dfp["id"].values)
th.insert(1, "fuente", fuentes)
th.insert(2, "año", años)
th.insert(3, "mes", meses)
th.insert(4, "es_breve", breve)
th.insert(5, "es_carta", dfp["es_carta"].values)
th["ok"] = OK
th["topico_dominante"] = np.where(OK, T_ALL.argmax(1), -1)
th["topico_dominante_agenda"] = np.where(OK_AG, np.array(KEEP)[T_AG.argmax(1)], -1)
th["en_ajuste"] = False
th.loc[idx_fit, "en_ajuste"] = True
th["en_heldout"] = False
th.loc[idx_ho, "en_heldout"] = True
th.to_parquet(OUT / "theta_documentos.parquet", index=False)

resumen = {
    "config": cfg.a_dict(),
    "tiempos_s": TIEMPOS,
    "n_docs": len(dfp),
    "n_ajuste": len(idx_fit),
    "n_heldout": len(idx_ho),
    "umbral_docs_mes": UMB.rename(index=ag.CORTO).to_dict(),
    "n_plurales": len(MAPA),
    "lista_blanca": sorted(set(LB) & cp),
    "estrategia_bigramas": ESTRATEGIA,
    "motivo_bigramas": motivo_b,
    "K": K_STAR,
    "motivo_K": motivo_k,
    "semilla_medoide": MED,
    "metricas_K_elegido": agg_k.loc[K_STAR, cols_k].to_dict(),
    "residuales": {NOMBRE[k]: TOP.motivo[k] for k in RESID},
    "docs_sin_topico": int((~OK).sum()),
    "estabilidad_agenda": RES_TR.set_index("serie").to_dict(orient="index"),
    "js_entre_fuentes": {
        "js": REF.js.mean(),
        "ruido": REF.nulo_media.mean(),
        "exceso": REF_EXC,
    },
    "cambios_series": S4,
    "cambios_formato": FORMATO,
    "cambios_contraste": CONTRASTE,
    "cambios_votantes": VOT,
    "robustez": ROB[["año", "veredicto", "soporte", "series_formato"]].to_dict(
        orient="records"
    ),
    "consenso": CONSENSO,
    "eventos": LBE,
}
ag.a_json(resumen, OUT / "resumen_08.json")
print(sorted(p.name for p in OUT.iterdir()))
print(
    json.dumps(
        {
            k: resumen[k]
            for k in [
                "estrategia_bigramas",
                "K",
                "motivo_K",
                "consenso",
                "eventos",
                "tiempos_s",
            ]
        },
        indent=2,
        ensure_ascii=False,
        default=str,
    )
)